In [3]:
import os
import math
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import logging

logger = logging.getLogger(name=__name__)



repo_root = Path.cwd()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

import src.auxFunctions as auxFunctions
import src.inputMechanicalParametersModel2 as MechanicalParams2
import src.vertexModel2 as vertexModel2
import src.auxFunctionsExpansion as auxFunctionsExpansion


In [ ]:
def collapse_single_edge(cellmap, geom, energyContributions_model, edge_id):
    """
    Collapses a single specified edge by merging its two vertices into one.
    The new vertex is placed at the midpoint of the original edge.
    
    Parameters:
    -----------
    cellmap : object
        The cellmap containing vertex and edge DataFrames
    geom : object
        Geometry handler
    energyContributions_model : object
        Energy model for the system
    edge_id : int
        ID of the edge to collapse (must be an inside edge, not boundary)
    
    Returns:
    --------
    cellmap : object
        Updated cellmap after edge collapse
    """
    
    logger.info(f"Collapsing edge: {edge_id}")
    
    # Verifying edge exists
    if edge_id not in cellmap.edge_df.index:
        raise ValueError(f"Edge {edge_id} not found in cellmap.edge_df")
    
    # Finding the two vertices belonging to the edge
    edge_row = cellmap.edge_df.loc[edge_id]
    v1 = edge_row['srce']
    v2 = edge_row['trgt']
    
    logger.info(f"Merging vertices {v1} and {v2}")
    
    # Verifying both vertices exist
    if v1 not in cellmap.vert_df.index or v2 not in cellmap.vert_df.index:
        raise ValueError(f"Vertex {v1} or {v2} not found in cellmap.vert_df")
    
    # Calculating midpoint coordinates along the chosen edge 
    x1 = cellmap.vert_df.loc[v1, 'x']
    y1 = cellmap.vert_df.loc[v1, 'y']
    x2 = cellmap.vert_df.loc[v2, 'x']
    y2 = cellmap.vert_df.loc[v2, 'y']
    
    midpoint_x = (x1 + x2) / 2
    midpoint_y = (y1 + y2) / 2
    
    # Creating a new vertex at the midpoint
    new_vertex_id = max(cellmap.vert_df.index) + 1 if not cellmap.vert_df.empty else 0
    new_vertex_data = cellmap.vert_df.loc[v1].copy()
    new_vertex_data['x'] = midpoint_x
    new_vertex_data['y'] = midpoint_y
    new_vertex_data['new_vert_id'] = np.nan   # ensures new_vert_ids assigns a novel one later on, not v1's old ID
    cellmap.vert_df.loc[new_vertex_id] = new_vertex_data
    
    # Rewiring edges by vertex proximity
    cellmap.edge_df.loc[cellmap.edge_df["srce"] == v1, "srce"] = new_vertex_id
    cellmap.edge_df.loc[cellmap.edge_df["srce"] == v2, "srce"] = new_vertex_id
    cellmap.edge_df.loc[cellmap.edge_df["trgt"] == v1, "trgt"] = new_vertex_id
    cellmap.edge_df.loc[cellmap.edge_df["trgt"] == v2, "trgt"] = new_vertex_id
    
    # Deleting the collapsed edge and its parallel edge
    parallel_edges = cellmap.edge_df[
        ((cellmap.edge_df["srce"] == v1) & (cellmap.edge_df["trgt"] == v2)) |
        ((cellmap.edge_df["srce"] == v2) & (cellmap.edge_df["trgt"] == v1))
    ].index.tolist()
    
    edges_to_delete = [edge_id] + parallel_edges
    cellmap.edge_df.drop(edges_to_delete, inplace=True, errors='ignore')
    
    # Deleting old vertices
    cellmap.vert_df.drop([v1, v2], inplace=True, errors='ignore')
    
    # Removing self-loops (safety mechanism, shouldn't exist)
    cellmap.edge_df = cellmap.edge_df[cellmap.edge_df["srce"] != cellmap.edge_df["trgt"]]
    
    # Removing duplicate edges (if many consecutive collapses form any)
    cellmap.edge_df = cellmap.edge_df.drop_duplicates(subset=['srce', 'trgt'])
    
    # Resetting indices and updating geometry
    cellmap.reset_index()
    
    # Updating active vertices
    if hasattr(cellmap, 'active_verts'):
        cellmap.active_verts = list(cellmap.vert_df.index)
    
    geom.update_all(cellmap)
    
    # Recomputing energy and relaxing the model post edge contraction
    energyContributions_model.compute_energy(cellmap)
    [cellmap, geom, model_H, history_H, solver] = vertexModel2.solveEuler(
        cellmap, geom, energyContributions_model, endTime=40
    )
    
    return cellmap

In [5]:
def split_vertex(cellmap, chosen_vertex, geom, energyContributions_model, distance, retry_attempts=3):
    """
    Safely divide a vertex by creating a nearby new vertex and rewiring edges.
    On failure, rollback and retry up to `retry_attempts` times.

    Returns:
        (cellmap, chosen_vertex, new_vert_index, new_edge_index, opposite_edge_index)
        or (cellmap, chosen_vertex, None, None, None) if all attempts fail.
    """

    original_vert_df = cellmap.vert_df.copy()
    original_edge_df = cellmap.edge_df.copy()

    new_vert_index = None
    new_edge_index = None
    opposite_edge_index = None

    attempt = 0
    while attempt < retry_attempts:
        try:
            # Creating temorary cellmap copies to allow rollback
            temp_vert_df = cellmap.vert_df.copy()
            temp_edge_df = cellmap.edge_df.copy()

            # Finding connected edges to the chosen vertex
            connected_edges = temp_edge_df[
                (temp_edge_df['srce'] == chosen_vertex) |
                (temp_edge_df['trgt'] == chosen_vertex)
            ].copy()

            # Creating new vertex inside a specified radius from chosen vertex
            new_vert_data = temp_vert_df.loc[chosen_vertex].copy()
            angle = np.random.uniform(0, 2*np.pi)
            dx = distance * np.cos(angle)
            dy = distance * np.sin(angle)
            new_vert_data[cellmap.coords] = temp_vert_df.loc[chosen_vertex, cellmap.coords] + [dx, dy]

            new_vert_index = int(temp_vert_df.index.max()) + 1
            temp_vert_df.loc[new_vert_index] = new_vert_data
    
        
            # Creating a new pair of edges
            # Template: first connected edge (copies its all mechanical properties)

            source_edge = connected_edges.iloc[0]

            template = source_edge.copy()

            new_edge_index = int(temp_edge_df.index.max()) + 1
            opposite_edge_index = new_edge_index + 1

            new_edge = template.copy()
            new_edge["srce"], new_edge["trgt"] = chosen_vertex, new_vert_index
            new_edge["face"] = np.nan

            opposite_edge = template.copy()
            opposite_edge["srce"], opposite_edge["trgt"] = new_vert_index, chosen_vertex
            opposite_edge["face"] = np.nan

            temp_edge_df.loc[new_edge_index] = new_edge
            temp_edge_df.loc[opposite_edge_index] = opposite_edge

            # Reassigning original edges to closer vertex
            chosen_xy = temp_vert_df.loc[chosen_vertex, cellmap.coords].values.astype(float)
            new_xy    = temp_vert_df.loc[new_vert_index, cellmap.coords].values.astype(float)

            reassigned_edges_to_new_vertex = []
            for e_idx, e in connected_edges.iterrows():
                if e_idx in (new_edge_index, opposite_edge_index):
                    continue
                other = e['trgt'] if e['srce'] == chosen_vertex else e['srce']
                other_xy = temp_vert_df.loc[other, cellmap.coords].values.astype(float)

                d_chosen = np.linalg.norm(other_xy - chosen_xy)
                d_new    = np.linalg.norm(other_xy - new_xy)

                if d_new < d_chosen:
                    reassigned_edges_to_new_vertex.append(e_idx)
                    if e['srce'] == chosen_vertex:
                        temp_edge_df.loc[e_idx, 'srce'] = new_vert_index
                    else:
                        temp_edge_df.loc[e_idx, 'trgt'] = new_vert_index

            # Identifying "open" faces (whose edge chains don't close)
            open_faces = []
            for face_id, group in temp_edge_df.groupby('face'):
                verts = list(group[['srce', 'trgt']].itertuples(index=False, name=None))
                if not verts:
                    continue

                # try to follow the loop
                chain = [verts[0][0], verts[0][1]]
                used = {0}
                while True:
                    extended = False
                    for i, (s, t) in enumerate(verts):
                        if i in used:
                            continue
                        if chain[-1] == s:
                            chain.append(t)
                            used.add(i)
                            extended = True
                            break
                        elif chain[-1] == t:
                            chain.append(s)
                            used.add(i)
                            extended = True
                            break
                    if not extended:
                        break

                # if loop doesn't close, this face is "open"
                if chain[0] != chain[-1]:
                    open_faces.append(face_id)

            # filtering to only faces touching the new vertex
            open_faces_touching_new = []
            for f in open_faces:
                verts_f = temp_edge_df[temp_edge_df['face'] == f][['srce', 'trgt']].values.ravel()
                if new_vert_index in verts_f or chosen_vertex in verts_f:
                    open_faces_touching_new.append(f)

            print("Open faces touching new vertex:", open_faces_touching_new)

            if len(open_faces_touching_new) != 2:
                raise ValueError(
                    f"Expected 2 open faces, found {len(open_faces_touching_new)}: {open_faces_touching_new}"
                )

            # Finding which new edge closes which open face (needs correct directionality)
            for f in open_faces_touching_new:
                f_edges = temp_edge_df[temp_edge_df['face'] == f]

                # collecting src/trgt vertices
                srces = list(f_edges['srce'].astype(int))
                trgts = list(f_edges['trgt'].astype(int))

                # imbalance: start and end
                start_candidates = [v for v in srces if v not in trgts]
                end_candidates   = [v for v in trgts if v not in srces]

                if len(start_candidates) == 1 and len(end_candidates) == 1:
                    start = start_candidates[0]
                    end   = end_candidates[0]
                    needed_edge = (end, start)  # must go end→start to close loop

                    new_pair      = (int(temp_edge_df.loc[new_edge_index, 'srce']),
                                     int(temp_edge_df.loc[new_edge_index, 'trgt']))
                    opposite_pair = (int(temp_edge_df.loc[opposite_edge_index, 'srce']),
                                     int(temp_edge_df.loc[opposite_edge_index, 'trgt']))

                    if new_pair == needed_edge:
                        temp_edge_df.loc[new_edge_index, 'face'] = f
                    elif opposite_pair == needed_edge:
                        temp_edge_df.loc[opposite_edge_index, 'face'] = f
                    else:
                        print(f"Face {f}: expected {needed_edge}, "
                              f"but new={new_pair}, opp={opposite_pair}")
                        
            # Verifying both faces are closed in directed edge cycles; if not, swap once and recheck

            def _face_closes(df, face_id):
                sub = df[df['face'] == face_id][['srce', 'trgt']]
                # in==out at every vertex
                outc = sub['srce'].value_counts()
                inc  = sub['trgt'].value_counts()
                verts = set(outc.index) | set(inc.index)
                for v in verts:
                    if outc.get(v, 0) != inc.get(v, 0):
                        return False
                # follow edges as a walk using srce->trgt
                start = int(sub.iloc[0]['srce'])
                cur = start
                used = set()
                for _ in range(len(sub)):
                    nxt = sub[~sub.index.isin(used) & (sub['srce'] == cur)]
                    if nxt.empty:
                        return False
                    eidx = nxt.index[0]
                    used.add(eidx)
                    cur = int(sub.loc[eidx, 'trgt'])
                return cur == start and len(used) == len(sub)

            face_new = int(temp_edge_df.loc[new_edge_index, 'face'])
            face_opp = int(temp_edge_df.loc[opposite_edge_index, 'face'])

            ok_new = _face_closes(temp_edge_df, face_new)
            ok_opp = _face_closes(temp_edge_df, face_opp)

            if not (ok_new and ok_opp):
                # try swapping faces between the two new edges once
                temp_edge_df.loc[new_edge_index, 'face'], temp_edge_df.loc[opposite_edge_index, 'face'] = face_opp, face_new
                face_new, face_opp = face_opp, face_new
                ok_new = _face_closes(temp_edge_df, face_new)
                ok_opp = _face_closes(temp_edge_df, face_opp)

            if not (ok_new and ok_opp):
                raise ValueError(
                    f"Face closure failed after assignment: "
                    f"new→face {face_new} ok={ok_new}, opp→face {face_opp} ok={ok_opp}"
            )

            # Committing temp results
            cellmap.vert_df = temp_vert_df
            cellmap.edge_df = temp_edge_df

            # Updating geometry
            geom.update_all(cellmap)
            cellmap.reset_topo()
            cellmap.reset_index()

            # Relaxing the model
            energyContributions_model.compute_energy(cellmap)
            [cellmap, geom, model_H, history_H, solver] = vertexModel2.solveEuler(
                cellmap, geom, energyContributions_model, endTime=40
            )

            print(f" Successfully divided vertex {chosen_vertex} → new vertex {new_vert_index}")
            return cellmap, chosen_vertex, new_vert_index, new_edge_index, opposite_edge_index

        except Exception as e:
            print(f" split_vertex attempt {attempt+1}/{retry_attempts} failed: {e}")
            # rollback original state
            cellmap.vert_df = original_vert_df.copy()
            cellmap.edge_df = original_edge_df.copy()
            geom.update_all(cellmap)
            cellmap.reset_topo()
            cellmap.reset_index()
            attempt += 1

    print("Failed to divide after multiple attempts.")
    return cellmap, chosen_vertex, None, None, None


In [ ]:
def run_expansion_simulation(
    cellmap_start,
    geom,
    energyContributions_model,
    # Features to include in expansion
    enable_detachment=False,
    enable_divisions=False,
    enable_collapses=False,
    # Directory for saving 
    output_dir="expansion_simulation",
    # How long to expand the tissue for
    total_steps=15000,
    steps_per_cycle=500,
    # Selecting amount of apoptosis
    collapse_fraction_per_remodel=0.0005,
    relax_after_collapse=10,
    # Selecting threshold for division
    edge_sum_threshold=7,
    division_distance=0.01,
    relax_after_division=10,
    # Selecting line tension relaxation
    tension_decrease_factor=0.4,
    # Selecting the fraction of edges to relax
    relax_tension_fraction=0.5,
    # Selecting factors for change in prefered area and area elasticity 
    pressure_increase_factor=8,
    area_elasticity_value=25,
    # Specifying capsule remodelling to allow expansion
    capsule_tension=300,
    capsule_viscosity=10000,
    # Limit cell cycle events for the inside of the tissue, to avoid boundary effects
    boundary_layers=5,
    # Plotting
    xlim=(-30, 70),
    ylim=(-30, 70),
):
    """
    Run tissue expansion simulation with optional:
    - enable_detachment = ECM's preferred length is reset to each edge's actual length at the start of expansion
    - enable_divisions = allowing cell proliferation if treshold is reached
    - enable_collapses = allowing apoptosis in the model through cell contraction
    - relax_tension_fraction = if <1 causes spatially heterogeneous relaxing of edges
    """

    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, "simulation_stats.json")

    # Building simulation description
    features = []
    if enable_detachment:
        features.append("detachment")
    if enable_divisions:
        features.append(f"divisions_thr{edge_sum_threshold}")
    if enable_collapses:
        features.append(f"collapses_{collapse_fraction_per_remodel:.4f}")
    
    if relax_tension_fraction > 0:
        features.append(f"relax_{int(relax_tension_fraction*100)}%_edges")
    
    simulation_desc = " + ".join(features) if features else "basic"

    cellmap = cellmap_start.copy()

    # Creating columns in vert_df to track vertex division 

    v = cellmap.vert_df
    if "new_vert_id" not in v.columns:
        v["new_vert_id"] = np.arange(len(v), dtype=int)
    if "parent_vert_id" not in v.columns:
        v["parent_vert_id"] = np.nan
    if "birth_step" not in v.columns:
        v["birth_step"] = 0
    if "divided_step" not in v.columns:
        v["divided_step"] = np.nan
    next_new_vert_id = int(v["new_vert_id"].max()) + 1

    # Changing model boundary (immitating capsule remodelling)

    boundary_edges, boundary_faces, inside_edges, outside_edges, \
        inside_faces, inside_vertices, outside_vertices = \
        auxFunctions.identify_boundary_layers(cellmap, 1)

    cellmap.vert_df.loc[outside_vertices, "viscosity"] = capsule_viscosity
    cellmap.edge_df.loc[outside_edges, "line_tension"] = capsule_tension

    # Choosing random edges for tension relaxation (relax_tension_fraction)

    edf = cellmap.edge_df
    n_edges = len(edf)
    k = int(np.floor(relax_tension_fraction * n_edges))
    k = max(0, min(k, n_edges))

    if k > 0:
        chosen_edges = np.random.choice(edf.index.to_numpy(), size=k, replace=False)
        edf.loc[chosen_edges, "line_tension"] *= tension_decrease_factor
        print(f"Relaxed {k}/{n_edges} edges to {tension_decrease_factor:.2f}× original tension")

    # Setting preffered area and area elasticity

    cellmap.face_df["prefered_area"] *= pressure_increase_factor
    cellmap.face_df["area_elasticity"] = area_elasticity_value

    # ECM detachment, setting edge's prefered length to its current length 
    if enable_detachment:
        cellmap.edge_df["prefered_length"] = cellmap.edge_df["length"]

    # Creating tracking lists
    division_stats = []
    collapse_stats = []
    remodel_nverts_stats = []
    collapse_target_stats = []
    inside_edges_total_stats = []

    # Saving initial state
    print(f"\n{'='*60}")
    print(f"Starting simulation: {simulation_desc}")
    print(f"Output directory: {output_dir}")
    print(f"{'='*60}\n")

    stats_data = {
        "simulation_description": simulation_desc,
        "parameters": {
            "total_steps": total_steps,
            "steps_per_cycle": steps_per_cycle,
            "collapse_fraction_per_remodel": collapse_fraction_per_remodel,
            "edge_sum_threshold": edge_sum_threshold,
            "division_distance": division_distance,
            "relax_after_collapse": relax_after_collapse,
            "relax_after_division": relax_after_division,
            "tension_decrease_factor": tension_decrease_factor,
            "relax_tension_fraction": relax_tension_fraction,
            "pressure_increase_factor": pressure_increase_factor,
            "area_elasticity_value": area_elasticity_value,
            "capsule_tension": capsule_tension,
            "capsule_viscosity": capsule_viscosity,
            "boundary_layers": boundary_layers,
        },
        "enabled_features": {
            "detachment": enable_detachment,
            "divisions": enable_divisions,
            "collapses": enable_collapses,
        },
    }

    with open(json_path, "w") as f:
        json.dump(stats_data, f, indent=2)

    # Saving initial checkpoint
    with open(os.path.join(output_dir, "checkpoint_000000.pkl"), "wb") as f:
        pickle.dump(cellmap, f)

    # Saving initial visualization
    fig, ax = auxFunctions.view(cellmap, geom, show_axes=True, xlim=xlim, ylim=ylim)
    plt.title(f"Step 0 | {simulation_desc}", fontsize=10)
    plt.savefig(os.path.join(output_dir, "progress_000000.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Helper functions

    def relax(cellmap, nsteps: int):
        """Run mechanical relaxation without topological changes."""
        if nsteps <= 0:
            return cellmap
        energyContributions_model.compute_energy(cellmap)
        cellmap_out, _, _, _, _ = vertexModel2.solveEuler(
            cellmap, geom, energyContributions_model, int(nsteps)
        )
        return cellmap_out

    def ensure_new_vert_ids(cellmap, next_id):
        """Assign unique IDs to any new vertices."""
        vdf = cellmap.vert_df
        if "new_vert_id" not in vdf.columns:
            vdf["new_vert_id"] = np.nan
        missing = vdf["new_vert_id"].isna()
        if missing.any():
            n = int(missing.sum())
            vdf.loc[missing, "new_vert_id"] = np.arange(next_id, next_id + n, dtype=int)
            next_id += n
        return next_id

    # Main simulation loop

    for step in range(0, total_steps, steps_per_cycle):
        current_step = step + steps_per_cycle
        print(f"\n--- Cycle {current_step}/{total_steps} ---")

        # Re-identifying model boundary
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, boundary_layers)

        current_divisions = 0
        current_collapses = 0

        # Carrying out edge collapses (apoptosis)

        if enable_collapses:
            valid_inside_edges = [e for e in inside_edges if e in cellmap.edge_df.index]
            n_inside = len(valid_inside_edges)
            target_collapse = int(math.floor(collapse_fraction_per_remodel * n_inside))
            target_collapse = max(0, min(target_collapse, n_inside))

            inside_edges_total_stats.append(n_inside)
            collapse_target_stats.append(target_collapse)

            if target_collapse > 0:
                edges_to_collapse = np.random.choice(valid_inside_edges, size=target_collapse, replace=False)
                for edge in edges_to_collapse:
                    try:
                        cellmap = collapse_single_edge(
                            cellmap, geom, energyContributions_model, edge
                        )
                        current_collapses += 1
                    except Exception as e:
                        print(f"  Warning: collapse failed for edge {edge}: {e}")

            print(f"  Collapses: {current_collapses}/{target_collapse}")

            cellmap.reset_index()
            cellmap.reset_topo()
            next_new_vert_id = ensure_new_vert_ids(cellmap, next_new_vert_id)

            if relax_after_collapse > 0:
                cellmap = relax(cellmap, relax_after_collapse)
                cellmap.reset_index()
                cellmap.reset_topo()

        # Re-identifying boundaries after collapses
        boundary_edges, boundary_faces, inside_edges, outside_edges, \
            inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(cellmap, boundary_layers)

        # Carrying out division (splitting vertices connected to edge length > threshold)

        if enable_divisions:
            # Finding vertices to divide using edge_sum_threshold
            vertices_to_divide = []
            for v in inside_vertices:
                if v not in cellmap.vert_df.index:
                    continue
                # Identifying connected edges to v
                connected_edges = cellmap.edge_df[
                    (cellmap.edge_df["srce"] == v) | (cellmap.edge_df["trgt"] == v)
                ]
                edge_sum = 0
                for edge_idx, edge in connected_edges.iterrows():
                    length = connected_edges.loc[edge_idx, 'length']
                    edge_sum = edge_sum + length
               
                
                if edge_sum > edge_sum_threshold:
                    vertices_to_divide.append(v)
            
            remodel_nverts_stats.append(len(vertices_to_divide))
            print(f"  Vertices above threshold: {len(vertices_to_divide)}")

            if len(vertices_to_divide) > 0:
                divided_count = 0
                
                for v in vertices_to_divide:
                    try:
                        cellmap, parent_vert, new_vert, new_edge, opp_edge = split_vertex(
                            cellmap, v, geom, energyContributions_model, division_distance, retry_attempts=3
                        )
                        
                        if new_vert is not None:
                            # Recording lineage
                            parent_id = int(cellmap.vert_df.loc[parent_vert, "new_vert_id"])
                            cellmap.vert_df.loc[parent_vert, "divided_step"] = current_step
                            cellmap.vert_df.loc[new_vert, "new_vert_id"] = next_new_vert_id
                            cellmap.vert_df.loc[new_vert, "parent_vert_id"] = parent_id
                            cellmap.vert_df.loc[new_vert, "birth_step"] = current_step
                            next_new_vert_id += 1
                            divided_count += 1
                            
                    except Exception as e:
                        print(f"  Warning: division failed for vertex {v}: {e}")
                
                current_divisions = divided_count

                # Saving post-division checkpoint
                with open(os.path.join(output_dir, f"checkpoint_{current_step:06d}_postdiv.pkl"), "wb") as f:
                    pickle.dump(cellmap, f)

            print(f"  Divisions: {current_divisions}")

            if relax_after_division > 0:
                cellmap = relax(cellmap, relax_after_division)
                cellmap.reset_index()
                cellmap.reset_topo()

        # Relaxing model post apoptosis and divisions
        used_steps = 0
        if enable_collapses:
            used_steps += relax_after_collapse
        if enable_divisions:
            used_steps += relax_after_division

        remaining_steps = steps_per_cycle - used_steps
        if remaining_steps < 0:
            raise ValueError("relax_after_collapse + relax_after_division exceeds steps_per_cycle")

        if remaining_steps > 0:
            cellmap = relax(cellmap, remaining_steps)

        # Recording statistics
        division_stats.append(current_divisions)
        collapse_stats.append(current_collapses)

        # Sasving checkpoint and visualisation
        with open(os.path.join(output_dir, f"checkpoint_{current_step:06d}.pkl"), "wb") as f:
            pickle.dump(cellmap, f)

        fig, ax = auxFunctions.view(cellmap, geom, show_axes=True, xlim=xlim, ylim=ylim)
        title = f"Step {current_step} | {simulation_desc} | Div:{current_divisions} Col:{current_collapses}"
        plt.title(title, fontsize=10)
        plt.savefig(os.path.join(output_dir, f"progress_{current_step:06d}.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)

        # Updating JSON statistics
        with open(json_path, "r") as f:
            existing_data = json.load(f)

        existing_data["last_saved_step"] = current_step
        existing_data["statistics"] = {
            "division_stats": division_stats if enable_divisions else None,
            "collapse_stats": collapse_stats if enable_collapses else None,
            "remodel_nverts_stats": remodel_nverts_stats,
            "collapse_target_stats": collapse_target_stats if enable_collapses else None,
            "inside_edges_total_stats": inside_edges_total_stats if enable_collapses else None,
        }

        with open(json_path, "w") as f:
            json.dump(existing_data, f, indent=2)

    return cellmap, division_stats, collapse_stats

In [7]:
# Initializing cellmap, geometry, and energy contributions model
cellmap_init, geom, energyContributions_model = vertexModel2.initialize(40)
cellmap_init = MechanicalParams2.update(cellmap_init)

boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = auxFunctions.identify_boundary_layers(cellmap_init, 1)

# Changing outside vertices mechanics
high_viscosity_value = 200000  
for vertex_id in outside_vertices:
    if vertex_id in cellmap_init.vert_df.index:
        cellmap_init.vert_df.at[vertex_id, "viscosity"] = high_viscosity_value

        
# Allowing model to relax
energyContributions_model.compute_energy(cellmap_init)
[cellmap_init, geom, energyContributions_model, history_new, solver1] = vertexModel2.solveEuler(cellmap_init, geom, energyContributions_model, 20)


CGAL-based mesh generation utilities not found, you may need to install CGAL and build from source
C++ extensions are not available for this version
Topology changed!


In [ ]:
cellmap, division_stats, collapse_stats = run_expansion_simulation(
    cellmap_init,
    geom,
    energyContributions_model,
    # Features to include in expansion
    enable_detachment=True,
    enable_divisions=True,
    enable_collapses=True,
    # Directory for saving 
    output_dir="expansion_simulation_test",
    # How long to expand the tissue for
    total_steps=15000,
    steps_per_cycle=500,
    # Selecting amount of apoptosis
    collapse_fraction_per_remodel=0.0005,
    relax_after_collapse=10,
    # Selecting threshold for division
    edge_sum_threshold=7,
    division_distance=0.01,
    relax_after_division=10,
    # Selecting line tension relaxation
    tension_decrease_factor=0.4,
    # Selecting the fraction of edges to relax
    relax_tension_fraction=1,
    # Selecting factors for change in prefered area and area elasticity 
    pressure_increase_factor=8,
    area_elasticity_value=25,
    # Specifying capsule remodelling to allow expansion
    capsule_tension=300,
    capsule_viscosity=10000,
    # Limit cell cycle events for the inside of the tissue, to avoid boundary effects
    boundary_layers=5,
    # Plotting
    xlim=(-30, 70),
    ylim=(-30, 70),
)

Relaxed 8114/8114 edges to 0.40× original tension

Starting simulation: detachment + divisions_thr3.5 + collapses_0.0005 + relax_100%_edges
Output directory: expansion_simulation_test


--- Cycle 500/15000 ---
Topology changed!
Topology changed!
  Collapses: 2/2
Topology changed!
  Vertices above threshold: 968
Open faces touching new vertex: [768.0, 805.0]
Topology changed!
 Successfully divided vertex 2.0 → new vertex 2752
Open faces touching new vertex: [805.0, 806.0]
Topology changed!
 Successfully divided vertex 3.0 → new vertex 2753
Open faces touching new vertex: [581.0, 618.0]
Topology changed!
 Successfully divided vertex 7.0 → new vertex 2754
Open faces touching new vertex: [284.0, 322.0]
Topology changed!
 Successfully divided vertex 8.0 → new vertex 2755
Open faces touching new vertex: [289.0, 326.0]
Topology changed!
 Successfully divided vertex 12.0 → new vertex 2756
Open faces touching new vertex: [921.0, 958.0]
Topology changed!
 Successfully divided vertex 26.0 → new v

KeyboardInterrupt: 